# Pika-style Phone Video Generator (Wan2GP backend)

Run every cell top to bottom on a **GPU runtime** (Runtime > Change runtime type > GPU).

**Real timing on a free Colab T4 (measured):** the `Wan 2.2 TextImage2Video FastWan`
model produces roughly a **5-second 480p clip in about 8 minutes**. Plan your
`max_scenes` accordingly — see the table in Cell 9 before you hit Generate.


In [ ]:
#@title 1. Confirm GPU runtime
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. Runtime > Change runtime type > GPU, then rerun.'
    ) from exc


In [ ]:
#@title 2. Workspace paths + optional Google Drive persistence
from pathlib import Path

USE_GOOGLE_DRIVE_DATA = True  #@param {type:"boolean"}

DRIVE_MOUNT_POINT = Path('/content/drive')
WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
EPHEMERAL_DATA_ROOT = Path('/content/Wan2GP-data').resolve()
PERSISTENT_DATA_ROOT = (DRIVE_MOUNT_POINT / 'MyDrive' / 'Wan2GP-data').resolve()

if USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    WAN_DATA_ROOT = PERSISTENT_DATA_ROOT
else:
    WAN_DATA_ROOT = EPHEMERAL_DATA_ROOT

for sub in ('ckpts', 'loras', 'outputs', 'cache'):
    (WAN_DATA_ROOT / sub).mkdir(parents=True, exist_ok=True)

WAN_CKPTS_DIR = WAN_DATA_ROOT / 'ckpts'
WAN_LORAS_DIR = WAN_DATA_ROOT / 'loras'
WAN_OUTPUTS_DIR = WAN_DATA_ROOT / 'outputs'
WAN_CACHE_DIR = WAN_DATA_ROOT / 'cache'

print(f'Data root: {WAN_DATA_ROOT}')


In [ ]:
#@title 3. Clone Wan2GP and link its data folders to your chosen storage
import subprocess

if not WAN2GP_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/deepbeepmeep/Wan2GP.git', str(WAN2GP_ROOT)], check=True)
else:
    print('Wan2GP already cloned, pulling latest...')
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)

def link(repo_sub, data_dir):
    target = WAN2GP_ROOT / repo_sub
    if target.is_symlink():
        return
    if target.exists():
        import shutil
        for item in target.iterdir():
            shutil.move(str(item), str(data_dir / item.name))
        target.rmdir()
    target.symlink_to(data_dir, target_is_directory=True)

link('ckpts', WAN_CKPTS_DIR)
link('loras', WAN_LORAS_DIR)
link('outputs', WAN_OUTPUTS_DIR)
print('Wan2GP checkpoints/loras/outputs now point at your chosen storage.')


In [ ]:
#@title 4. System dependencies (ffmpeg + libs)
# Colab's preinstalled ffmpeg sometimes lacks -fps_mode, which Wan2GP needs.
# Try apt first (fast); if a generation later fails on an ffmpeg option
# error, install a current static build from https://github.com/BtbN/FFmpeg-Builds
# and put it earlier on PATH.
import subprocess
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'], check=True)
print('System dependencies installed.')


In [ ]:
#@title 5. Python dependencies (PyTorch + Wan2GP + orchestrator)
import subprocess, sys

# Known-working combo for current Colab CUDA runtimes.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
                '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(WAN2GP_ROOT / 'requirements.txt')], check=True)

import os
REPO_URL = "https://github.com/<you>/<repo>.git"  #@param {type:"string"}
if not os.path.exists("/content/pika-video-generator"):
    subprocess.run(['git', 'clone', REPO_URL, '/content/pika-video-generator'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                '/content/pika-video-generator/requirements.txt'], check=True)
print('Dependencies installed.')


In [ ]:
#@title 5b. Headless matplotlib fix (needed for Wan2GP's preprocessing tools)
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
if target.exists():
    text = target.read_text()
    if "matplotlib.use('TkAgg')" in text:
        target.write_text(text.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')", 1))
        print('Patched matplotlib backend to Agg.')
    else:
        print('No patch needed.')
else:
    print('File not found - skipping (Wan2GP version may differ).')


In [ ]:
#@title 6. Clone Wav2Lip + download pretrained checkpoint
import subprocess, os
if not os.path.exists('/content/Wav2Lip'):
    subprocess.run(['git', 'clone', 'https://github.com/Rudrabha/Wav2Lip.git', '/content/Wav2Lip'], check=True)
os.makedirs('/content/Wav2Lip/checkpoints', exist_ok=True)
ckpt = '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'
if not os.path.exists(ckpt):
    subprocess.run(['wget', '-q', '-O', ckpt,
        'https://huggingface.co/spaces/aiswaryaravindkumar/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth'])
    if not os.path.exists(ckpt) or os.path.getsize(ckpt) < 1000000:
        print('WARNING: checkpoint download looks wrong/failed - manually place wav2lip_gan.pth in that folder.')
    else:
        print('Wav2Lip checkpoint ready.')
else:
    print('Checkpoint already present.')


In [ ]:
#@title 7. Launch Wan2GP headless (LOCAL ONLY - no --share, our own UI is the public one)
import subprocess, os, time

env = os.environ.copy()
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')

wan2gp_process = subprocess.Popen(
    ['python', 'wgp.py', '--listen', '--server-port', '7860', '--profile', '5'],
    cwd=str(WAN2GP_ROOT),
    env=env,
)
print('Starting Wan2GP on http://127.0.0.1:7860 ... waiting 40s')
time.sleep(40)
print('Wan2GP should be up now. Check the "* Running on local URL" line if you print wan2gp_process output.')


In [ ]:
#@title 8. Inspect Wan2GP's API (run once, confirm endpoint name)
import sys
sys.path.insert(0, "/content/pika-video-generator/src")
from wan2gp_client import inspect_api
inspect_api()
# Match what's printed here against WAN2GP_API_NAME in wan2gp_client.py
# and adjust that file if they differ.


## 9. Before you generate: pick a realistic scene count

Measured on a free Colab T4: **~8 minutes per 5-second 480p clip**, before
adding voice + lip-sync time on top. Rough totals (video generation only):

| Scenes | Video length | Est. generation time |
|---|---|---|
| 5  | ~25 sec | ~40 min |
| 10 | ~50 sec | ~80 min |
| 20 | ~100 sec | ~160 min (over 2.5 hrs) |
| 60 | ~5 min | ~8 hours (will NOT survive one free session) |

Start with **5-10 scenes**. A true 5-minute video on a free GPU is not
realistic in one sitting — either use Colab Pro for a faster GPU, or plan to
resume across multiple sessions (Drive persistence in Cell 2 helps here).


In [ ]:
#@title 10. Launch your custom Pika-style UI (open this link on your phone)
%cd /content/pika-video-generator/src
!python orchestrator_ui.py
